In [ ]:
# Operate from the parent directory
# This allows us to import modules from the parent directory
import os
os.chdir("..")

import torch
import matplotlib.pyplot as plt
from experimental_pose_encoder_model_extension.advanced_pose_encoder import AdvancedPoseEncoder
from dataset.dataset import *
from utils.animation.skeleton import Skeleton
import utils.utils as utils
import pickle
from torch.amp import autocast
import time

device = utils.get_device()

In [ ]:
dataset = GPUDataset(
    consolidated_file="dataset/genea2023_dataset/toy/main-agent/consolidated.npz",
    seq_length=2000, # For testing
    seed_length=0,
    batch_size=1,
    epoch_length=1,  # Set to 1 for testing purposes
    return_audio_frame_index=True,  # Set to True to return the audio frame index
)

model = AdvancedPoseEncoder.load_from_checkpoint("advanced_pose_encoder_ik_pca_64")

In [ ]:
# Import the viewer
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
animation_visualisation.init_visualization()
time.sleep(1.0) # Give the visualisation time to start before we send messages
animation_visualisation.send_character("ground_truth", (0.0,0.0,0.0), 0, "red")

In [ ]:
model.to(device)  # Move the model to the appropriate device
model.eval()  # Set the model to evaluation mode
# Use no gradient calculation for inference
with torch.no_grad():
    # Get a sample from the dataset
    # gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, full_audio_features, start_frame, file = next(iter(dataset))
    gesture_sequence, seed_gesture, audio_features, main_agent_id_one_hot, finger_availability, start_frames = next(iter(dataset))
    gesture_sequence = gesture_sequence.to(device)
    seed_gesture = seed_gesture.to(device)
    audio_features = audio_features.to(device)
    main_agent_id_one_hot = main_agent_id_one_hot.to(device)
    full_audio_features = dataset.audio.to(device)
    start_frame = start_frames[0].item()  # Extract the first element from the tensor

    with autocast(device_type=device.type, dtype=torch.bfloat16):
        # Decode the output using the autoencoder model
        gesture_sequence = gesture_sequence  # remove the batch dimension
        print("Gesture sequence dtype:", gesture_sequence.dtype)

        gesture_sequence_encoded = model.encode(gesture_sequence)

        print("Encoded gesture sequence dtype:", gesture_sequence_encoded.dtype)

        # Decode the output using the autoencoder model
        gesture_sequence_decoded = model.decode(gesture_sequence_encoded)

    # Undo the z-score normalization
    unnormalized_encoded_gesture_sequence = dataset.skeleton.denormalize_poses(gesture_sequence_decoded).squeeze(0).cpu()
    unnormalized_ground_truth_gesture_sequence = dataset.skeleton.denormalize_poses(gesture_sequence).squeeze(0).cpu()

    for i in range(unnormalized_encoded_gesture_sequence.shape[0]):
        animation_visualisation.send_pose(unnormalized_encoded_gesture_sequence[i], dataset.skeleton)
        animation_visualisation.send_pose(unnormalized_ground_truth_gesture_sequence[i], dataset.skeleton, "ground_truth")
        time.sleep(1.0/30.0) # # Assuming 30 FPS for the animation

In [ ]:
# Load std_pose and mean_pose from statistics.npz
with open("dataset/genea2023_dataset/trn/main-agent/consolidated_meta.pkl", 'rb') as f:
    skeleton: Skeleton = pickle.load(f)['skeleton']
    skeleton.set_device(device)

with torch.no_grad():
    # Generate two random poses and interpolate between them
    sample1 = torch.randn(1, model.z_dim).to(device)  # Sample random z-values
    pose1 = model.decode(sample1)

    # Get the first 64 samples from the dataset
    for i in range(64):
        
        sample2 = torch.randn(1, model.z_dim).to(device)  # Sample random z-values
        pose2 = model.decode(sample2)

        for alpha in torch.linspace(0, 1, 40):
            # Interpolate between the two poses
            x_reconstructed = (1 - alpha) * pose1 + alpha * pose2

            # unormalize the reconstructed pose
            x_reconstructed_unnormalized = skeleton.denormalize_poses(x_reconstructed)

            # Send the reconstructed pose to the viewer
            reconstructed_frame = animation_visualisation.add_pose_to_message(x_reconstructed_unnormalized.squeeze(0).cpu(), skeleton)
            animation_visualisation.send_message(reconstructed_frame)
        # animation_visualisation.send_frame(reconstructed_frame)

        # Update pose1 to pose2 for the next iteration
        pose1 = pose2

In [ ]:
import time
from IPython.display import clear_output
from utils.animation.skeleton import Skeleton

with torch.no_grad():

    # Load the dataset
    dataset = GPUDataset(
        consolidated_file= "dataset/genea2023_dataset/toy/main-agent/consolidated.npz",
        seq_length=1,
        seed_length=0,
        batch_size=1,
        epoch_length=64
    )

    skeleton: Skeleton = dataset.skeleton

    skeleton.set_device(device)

    # Get the first 64 samples from the dataset
    for i in range(64):
        pose, _, _, _ = [
            item.squeeze(0) for item in dataset[i]
        ]

        # pose to float # TODO: Probably dont do this
        pose = pose.float()

        # pass the pose through the model
        # x_reconstructed, mu, logvar = model(pose)
        x_encoded_mean, x_encoded_var = model.encode(pose, return_logvar=True)
        x_reconstructed = model.decode(x_encoded_mean)

        # unormalize the reconstructed pose
        x_reconstructed_unnormalized = skeleton.denormalize_poses(x_reconstructed)

        # unnormalize the original pose
        pose_unnormalized = skeleton.denormalize_poses(pose)

        # Draw a picture of the original and reconstructed pose
        plt.subplot(1, 4, 1)
        plt.imshow(pose.repeat(100,1).cpu().numpy(),cmap='gray')
        plt.title("Original Pose")
        plt.axis('off')
        # Write the mean of the pose
        plt.text(0, 20, f"Mean: {pose.mean().item():.2f}", color='white', fontsize=12)
        # Write the std of the pose
        plt.text(0, 40, f"Std: {pose.std().item():.2f}", color='white', fontsize=12)

        plt.subplot(1, 4, 2)
        plt.imshow(x_reconstructed.repeat(100,1).cpu().numpy(),cmap='gray')
        plt.title("Reconstructed Pose")
        plt.axis('off')
        # Write the mean of the pose
        plt.text(0, 20, f"Mean: {x_reconstructed.mean().item():.2f}", color='white', fontsize=12)
        # Write the std of the pose
        plt.text(0, 40, f"Std: {x_reconstructed.std().item():.2f}", color='white', fontsize=12)

        # Draw the encoded mean and variance
        plt.subplot(1, 4, 3)
        plt.imshow(x_encoded_mean.repeat(10,1).cpu().numpy(),cmap='gray')
        plt.title("Encoded Mean")
        plt.axis('off')
        plt.subplot(1, 4, 4)
        plt.imshow(x_encoded_var.repeat(10,1).cpu().numpy(),cmap='gray')
        plt.title("Encoded Variance")
        plt.axis('off')
        plt.tight_layout()

        plt.show()

        # Send the original pose to the viewer
        
        original_frame = animation_visualisation.add_pose_to_message(pose_unnormalized.squeeze(0).cpu(), skeleton)
        animation_visualisation.send_message(original_frame)
        # animation_visualisation.send_message(original_frame)

        print("Original pose sent to viewer")

        # wait for 1 seconds
        time.sleep(0.5)

        # Send the reconstructed pose to the viewer
        reconstructed_frame = animation_visualisation.add_pose_to_message(x_reconstructed_unnormalized.squeeze(0).cpu(), skeleton)
        animation_visualisation.send_message(reconstructed_frame)
        # animation_visualisation.send_frame(reconstructed_frame)

        print("Reconstructed pose sent to viewer")

        # wait for 3 seconds
        time.sleep(3)

        clear_output(wait=True)